In [ ]:
# NOTEBOOK NAME
# SimpleHovmoller.ipynb
# NOTEBOOK NAME

# imports
import numpy as np
import matplotlib as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

# for pretty colour maps
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# Load in a DEM (update path and variable name as needed)
DEMpath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'  # <- your DEM file
DEMdata = xr.open_dataset(DEMpath)
DEMelev = DEMdata['elevation']  # adjust if your var has a different name

In [ ]:
# HOVMOLLER LOADING WITH GRIDDED DATA

# ummmm, how many minutes are there between radar scans/ radar data times? please let me know that here to use for writing the times on the hovmoller grid
TimeIntervalMin = 5

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9

Altitude = 2000 # [m] height in the radar data (must be divisble by 500)

#Angle and Y-intercept of the slice
ViewingAngle = 120  # [Degrees]
ViewingYint  = 25   # [km] 

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# CHOOSE THE VARIABLE TO PLOT
Var  = 'Z'

# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
# [V]         'corrected_velocity'
# [AzSh]      'azshear'


# CHOOSE YOUR QUALITY CONTROL SETTINGS
# # taken from Aragon et al. 2024
# MinValidZDR = -4 # ZDR DOESNT WORK FOR MACKAY EARLY 2024 BECAUSE OF THE SOURCE RADAR DATA IN STORAGE
# MaxValidZDR =  4
MinValidRhoHV = 0.85



# USER CHOICE FOLLOW-ON SECTION

# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarFillValue = -32.0
    VarColourBar = make_ChadMapZ() # fetch a custom colour bar
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

NumTimes = int(1440 / 5)      # number of times in a day (288 sets of 5 min here)
NumSlicePoints = 300          # number of longitudes in the gridded radar data

# create an empty array to store what will be plotted in the hovmoller diagram
HovGrid      = np.full([NumTimes,NumSlicePoints], np.nan)
# create an empty array to store the times associated with each data point in the hovmoller diagram
HovGridTimes = np.full(NumTimes, np.datetime64('NaT'), dtype='datetime64[ns]')
# create an empty array to store the distances associated with each data point
HovGridDists = np.full([NumTimes,NumSlicePoints], np.nan)

# index of 5 min intervals
mini5 = -1
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
LoopStartTime='00:00'
LoopEndTime='23:55'
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    mini5 = mini5 + 1
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    # add a string of format hh:mm:ss for printing
    print('working on ' + RadarFileTimePrint)

    NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                            RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

    
    # try to load in the netcdf file and if it doesn't work, just keep going through the loop
    last_valid_time = np.datetime64(f'{YYYY}-{MM}-{DD}T00:00:00')  # pretend the last valid time was the start of the day for the first entry
    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
        HovGridTimes[mini5] = np.array(xgrid.time[0])
        last_valid_time = HovGridTimes[mini5]
    except FileNotFoundError:
        HovGrid[mini5, :] = np.full(NumSlicePoints, np.nan)
        HovGridDists[mini5,:] = np.full(NumSlicePoints, np.nan)
        last_valid_time = last_valid_time + np.timedelta64(TimeIntervalMin, 'm') # if there is no file for the time assume the time the radar reports is exactly 5 minutes after the last file
        HovGridTimes[mini5] = last_valid_time
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')


    # ROUGH QUALITY CONTROL SECTION
    # create a mask only where these quality control condtions are met
    ConditionGridA = xgrid['corrected_cross_correlation_ratio'] > MinValidRhoHV   # (y, x) boolean masks
    
    # I WOULD LIKE TO ADD CONDITIONS WITH DIFFERENTIAL REFLECTIVITY, BUT THIS RADAR DOES NOT HAVE VALID DATA YET
    # ConditionGridB = xgrid['corrected_differential_reflectivity'] > MinValidZDR
    # ConditionGridC = xgrid['corrected_differential_reflectivity'] < MaxValidZDR  
    
    ConditionGrid = ConditionGridA #* ConditionGridB * ConditionGridC   # combined boolean mask

    # apply the condtional mask to the variable array before plotting
    ValidVariableArray = xgrid[VarNameLong].where(ConditionGrid)
    PlottingArray = ValidVariableArray[0,np.where(xgrid.z == Altitude)[0][0],:,:]
    
    # HovGrid[mini5, :] = extract_line_slice( xgrid.corrected_reflectivity[0,np.where(xgrid.z == Altitude)[0][0],:,:], xgrid.x, xgrid.y, ViewingAngle, ViewingYint*1000, num_points=NumSlicePoints)[1]

    # HovGridDists[mini5,:] = extract_line_slice( xgrid.corrected_reflectivity[0,np.where(xgrid.z == Altitude)[0][0],:,:], xgrid.x, xgrid.y, ViewingAngle, ViewingYint*1000, num_points=NumSlicePoints)[0]


    HovGrid[mini5, :] = extract_line_slice( PlottingArray, xgrid.x, xgrid.y, ViewingAngle, ViewingYint*1000, num_points=NumSlicePoints)[1]

    HovGridDists[mini5,:] = extract_line_slice( PlottingArray, xgrid.x, xgrid.y, ViewingAngle, ViewingYint*1000, num_points=NumSlicePoints)[0]

    HovGridTimes[mini5] = np.array(xgrid.time[0])
    # print(np.array(xgrid.time[ np.where( (xgrid.azimuth == AziAngle) & (xgrid.elevation == EleAngle) ) ])[0])

# we just need to retrieve the coordinates once from some loaded net cdf, so might as well be the last one
HovGridLats = extract_line_slice(xgrid['lat'], xgrid.x, xgrid.y, ViewingAngle, ViewingYint*1000, num_points=NumSlicePoints)[1]
HovGridLons = extract_line_slice(xgrid['lon'], xgrid.x, xgrid.y, ViewingAngle, ViewingYint*1000, num_points=NumSlicePoints)[1]

In [ ]:
# HOVMOLLER PLOTTING

# Convert times to minutes since midnight of that day

# Build start-of-day as numpy datetime64 (UTC)
start_of_day = np.datetime64(RadarFileDatePrint + 'T00:00')
HovGridMins = (HovGridTimes - start_of_day).astype('timedelta64[m]').astype(float)

# Format minutes as HH:MM for y-axis tick labels
def minutes_to_hhmm(m, pos):
    m = int(m)
    h = m // 60
    mm = m % 60
    return f'{h:02d}:{mm:02d}'


# ── TOPOGRAPHY EXTRACTION FOR HOV SLICE ──────────────────────────────────────
LONdata = xr.DataArray(HovGridLons, dims=('points',))
LATdata = xr.DataArray(HovGridLats, dims=('points',))

# Interpolate DEM onto the 300 slice points
dem_hov = DEMelev.interp(lon=LONdata, lat=LATdata)
TerrainHovM  = dem_hov.values                          # elevation in metres

# PLOTTING REAL DATA HERE
# ── FIGURE WITH TWO PANELS ────────────────────────────────────────────────────
# Top panel (Hovmoller) is tall, bottom panel (topography) is short
fig, (ax, ax_topo) = plt.subplots(
    nrows=2, ncols=1,
    figsize=(14, 8),
    gridspec_kw={'height_ratios': [6, 1]},  # Hovmoller 6x taller than topo panel
    sharex=True                              # shared x-axis so they line up perfectly
)

# ── TOP PANEL: HOVMOLLER ──────────────────────────────────────────────────────
HovViewer = ax.pcolormesh(HovGridDists[0]*0.001, HovGridMins, HovGrid,
                          cmap=VarColourBar, norm=VarColourBar_norm)

cbar = plt.colorbar(HovViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
cbar.ax.set_ylim(VarMinVal, VarMaxVal)
cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, 10))

# Time flow arrow
ax.annotate(
    '',
    xy=(-0.08, 0.2),
    xytext=(-0.08, 0.8),
    xycoords='axes fraction',
    textcoords='axes fraction',
    arrowprops=dict(arrowstyle='->,head_width=0.4,head_length=0.5',
                    color='black', linewidth=2),
    annotation_clip=False,
)

ax.invert_yaxis()
ax.invert_xaxis()
ax.set_ylabel('Time (UTC)')
ax.yaxis.set_major_locator(mticker.MultipleLocator(120))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(minutes_to_hhmm))
ax.grid(True)

ax.set_title(VarName + ' (ρHV > ' + str(MinValidRhoHV) + ') Hovmoller Diagram\nFor ' + RadarSiteName + ' Radar on ' + \
             RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + \
             ' at ' + str(Altitude * 0.001) + ' km Altitude' + \
             '\nFor a slice looking ' + str(ViewingAngle) + '° (and ' + str(ViewingAngle+180) + \
             '°) from ' + str(ViewingYint) + ' km North of the Radar')

# ── BOTTOM PANEL: TOPOGRAPHY ──────────────────────────────────────────────────
SliceDistKM = HovGridDists[0] * 0.001   # x-axis distances in km

ax_topo.fill_between(SliceDistKM, 0, TerrainHovM, color=[0.3, 0.3, 0.3])
ax_topo.plot(SliceDistKM, TerrainHovM, color='black', linewidth=1)

ax_topo.set_ylabel('Elevation [m]')
ax_topo.set_xlabel('Distance Along Slice (Travelling East) [km]')
ax_topo.set_ylim(0,1200)
ax_topo.invert_xaxis()   # match the Hovmoller x-axis direction
ax_topo.grid(True)

plt.xlim([-150,150]) #[np.min(SliceDistKM), np.max(SliceDistKM)])
# change limit to be a maximum of 150 km away from the radar
plt.tight_layout()

# force the topo panel to match the Hovmoller panel width exactly
fig.canvas.draw()  # force matplotlib to finalise all positions first

# Get the position of the top (Hovmoller) panel
pos_top  = ax.get_position()
pos_topo = ax_topo.get_position()

# Keep the topo panel's y position and height, but steal the x and width from the top panel
ax_topo.set_position([pos_top.x0,    # same left edge
                      pos_topo.y0,   # keep its own vertical position
                      pos_top.width, # same width as Hovmoller panel
                      pos_topo.height])  # keep its own height

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/hov/' + RadarIDno + '/' + RadarFileDate + '/' + VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarName + '_hovGRID.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('Creating Folder: ' + SaveFolder)
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
plt.close()